<a href="https://colab.research.google.com/github/emanfatimaa05/urdu-ocr-codesaviours-si26-eman/blob/main/week4_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/emanfatimaa05/urdu-ocr-codesaviours-si26-eman.git
%cd urdu-ocr-codesaviours-si26-eman

Cloning into 'urdu-ocr-codesaviours-si26-eman'...
remote: Enumerating objects: 421, done.
remote: Counting objects: 100% (421/421), done.
remote: Compressing objects: 100% (409/409), done.
remote: Total 421 (delta 51), reused 222 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (421/421), 4.68 MiB | 2.25 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/urdu-ocr-codesaviours-si26-eman


In [4]:
!pip install transformers torch pillow pandas sentencepiece -q

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, ViTImageProcessor, RobertaTokenizer
from torch.optim import AdamW
from PIL import Image
import pandas as pd

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

from transformers import AutoTokenizer

image_processor_name = "microsoft/trocr-base-printed"
encoder_name = "google/vit-base-patch16-384"
decoder_name = "urduhack/roberta-urdu-small"

image_processor = ViTImageProcessor.from_pretrained(image_processor_name)
tokenizer = AutoTokenizer.from_pretrained(decoder_name)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)

model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(encoder_name, decoder_name)
model = model.to(device)

model.config.decoder_start_token_id = tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  347MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  507MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] RobertaForCausalLM LOAD REPORT from: urduhack/roberta-urdu-small
Key                                                                   | Status     | 
----------------------------------------------------------------------+------------+-
roberta.pooler.dense.bias                                             | UNEXPECTED | 
roberta.pooler.dense.weight                                           | UNEXPECTED | 
roberta.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
roberta.encoder.layer.{0...11}.crossattention.self.key.weigh

Model loaded successfully!
Model parameters: 241,079,584


In [13]:
class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128,
            truncation=True
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [14]:
dataset = UrduOCRDataset('data/labels.csv', processor)

torch.manual_seed(42)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 205 samples
Training samples: 164
Testing samples: 41


In [15]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

Training batches per epoch: 41
Ready to train!


In [20]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/5
------------------------------
  Batch 0/41 | Loss: 0.0169
  Batch 10/41 | Loss: 0.0255
  Batch 20/41 | Loss: 0.0201
  Batch 30/41 | Loss: 0.0195
  Batch 40/41 | Loss: 0.0296
Epoch 1 complete | Average Loss: 0.0274

Epoch 2/5
------------------------------
  Batch 0/41 | Loss: 0.0278
  Batch 10/41 | Loss: 0.0214
  Batch 20/41 | Loss: 0.0164
  Batch 30/41 | Loss: 0.0241
  Batch 40/41 | Loss: 0.0166
Epoch 2 complete | Average Loss: 0.0245

Epoch 3/5
------------------------------
  Batch 0/41 | Loss: 0.0277
  Batch 10/41 | Loss: 0.0189
  Batch 20/41 | Loss: 0.0214
  Batch 30/41 | Loss: 0.0150
  Batch 40/41 | Loss: 0.0296
Epoch 3 complete | Average Loss: 0.0189

Epoch 4/5
------------------------------
  Batch 0/41 | Loss: 0.0220
  Batch 10/41 | Loss: 0.0079
  Batch 20/41 | Loss: 0.0222
  Batch 30/41 | Loss: 0.0144
  Batch 40/41 | Loss: 0.0162
Epoch 4 complete | Average Loss: 0.0159

Epoch 5/5
------------------------------
  Batch 0/41 | Loss: 0.0059
  Batch 10/41 | Loss: 0.00

In [26]:
model.save_pretrained('urdu_ocr_finetuned')
processor.save_pretrained('urdu_ocr_finetuned')
print('Model saved!')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [24]:
!pip install jiwer -q
from jiwer import cer

model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0
all_preds = []
all_actuals = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values, max_new_tokens=64, num_beams=4)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            all_preds.append(pred.strip())
            all_actuals.append(actual.strip())
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Exact-Match Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')
print(f'Character Error Rate (CER): {cer(all_actuals, all_preds):.3f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.1 MB/s eta 0:00:00
=== Model Evaluation on Test Images ===

Predicted: فیس بک پر موجود جملے ڈاکٹر طارق چیمہ نامی صارف کی وال پر لکھے تھے جس کا عنوان انہوں نے دیا تھا 'آئی ایچ آئی سی کا عروج اور زوال: ایک کہانی جسے پاکستان کو فراموش نہیں کرنا چاہیے۔'
Actual: عظمیٰ بخاری نے کہا کہ 'پیپلز پارٹی کی پنجاب والی قیادت کا بارش میں زیادہ پنجاب حکومت پر فوکس ہے، یہ لوگ اپنی پاورٹی پر توجہ دیتے تو آج پنجاب میں پیپلز پارٹی کا یہ حال نہ ہوتا'۔

Predicted: امن ہر معاشرے کی ضرورت ہے
Actual: کراچی پاکستان کا سب سے بڑا شہر ہے

Predicted: وقت کی قدر کرنی چاہیے
Actual: محنت کامیابی کی کنجی ہے

Predicted: اخلاق موضوعات میں اسلوب کے سب سے اہم موضوعات میں سے ایک ہے، جس سے انسان کے اخلاق و کردار کی تشکیل ہوتی ہے۔
Actual: اسسٹنٹ ڈائریکٹر ایگریکلچر آن فارم واٹر مینجمنٹ

Predicted: اخلاق موضوعات میں اسلوب کے سب سے اہم موضوعات میں سے ایک ہے، جس سے انسان کے اخلاق و کردار کی تشکیل ہوتی ہے۔
Actual: نقیب نے مجھے طلب کیا میرے اثاثوں میں... ...صرف تم نکلے۔

Pre

In [25]:
# Test on a TRAINING image instead of test image, to see if it just memorized training data
model.eval()
with torch.no_grad():
    sample = train_dataset[0]
    pixel_values = sample['pixel_values'].unsqueeze(0).to(device)
    generated_ids = model.generate(pixel_values, max_new_tokens=64, num_beams=4)
    print("Predicted:", processor.batch_decode(generated_ids, skip_special_tokens=True)[0])
    print("Actual:", processor.tokenizer.decode(sample['labels'], skip_special_tokens=True))

Predicted: نادر اگاڑی صدر
Actual: نادر اگاڑی صدر


In [27]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained('/content/drive/MyDrive/SI26-urdu-ocr-model')
processor.save_pretrained('/content/drive/MyDrive/SI26-urdu-ocr-model')
print("Model saved to Drive!")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Drive!
